# Step 2: Process Features

## Traitement des attributs

In [ ]:
%load_ext autoreload
%autoreload 2


import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os

from method_a_buffer import extract_buffer_feature
from feature_query import make_feature_query

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm
import rasterio
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)


operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'


# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_pedestrian_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)

# Import des attributs
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]


Loading pedestrian segments...


In [2]:
attributs_info

,Class,meta_attribute,attribute,include_in_index,attribute_source_path,initial_weight,impact_attribut,file_name,geometry_type,method,how,value_column,buffer_size,filter_column,filter_values,crs,save_format
0,Sécurité,accident,accident,True,accident,0.3,defavorable,OTC_ACCIDENTS-SHP/OTC_ACCIDENTS.shp,point,A,count,NaN,10,filtered,1,2056,parquet
1,Sécurité,traffic,zone_apaisee,True,vitesse,0.5,favorable,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
2,Sécurité,traffic,zone_pietonne,True,vitesse,1.0,favorable,OTC_ZONE_MODERATION_TRAFIC-SHP/OTC_ZONE_MODERA...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
3,Sécurité,traffic,vitesse,True,vitesse,0.6,defavorable,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
4,Infrastructure,stationnement_genant,stationnement_genant,True,stationnement_genant,0.3,defavorable,SHP_FDP/FDP_STATIONNEMENT_GENANT_PIETON_CONTRA...,point,A,count,NaN,10,filtered,1,2056,parquet
5,Infrastructure,connectivite,connectivite,True,connectivite,0.7,favorable,NaN,line,A,sum,conn_branching_in_buffer,10,filtered,1,2056,parquet
6,Infrastructure,largeur_trottoir,largeur_trottoir,True,network_couche_OCT.shp,0.5,favorable,RP_final.shp,line,A,count,NaN,10,filtered,1,2056,parquet
7,Infrastructure,topographie,topographie,True,network_couche_OCT.shp,0.4,defavorable,RP_final.shp,line,A,sum,Pente,10,filtered,1,2056,parquet
8,Attractivité,eau,eau,True,eau,0.3,favorable,LCE_GRAPHE_EAU-SHP/LCE_GRAPHE_EAU.shp,line,A,count,NaN,10,filtered,1,2056,parquet
9,Attractivité,proximite,rez_actif,True,rez_actif,0.5,favorable,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,point,A,count,NaN,10,filtered,1,2056,parquet


In [3]:

# Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()

'''# --- ADD THIS BLOCK ---
# Sample a subset of the network for faster testing
sample_fraction = 0.7  # 70% of all segments, adjust as needed
segmented_net = segmented_net.sample(frac=sample_fraction, random_state=42)

print(f"Testing on a sample of {len(segmented_net)} segments out of the total network.")
# ----------------------'''

print('Boucle sur chaque attribut... peut prendre du temps (30 mn)')

# Boucle sur le derniers attributs
for _, row in attributs_info.iterrows():
    if row['include_in_index']:
        attribute_name = row['attribute']
        method = row['method']
        how = row['how']
        value_column = row['value_column']
        buffer_size = row['buffer_size']
        geometry_type = row['geometry_type']
        feature_query_expr = make_feature_query(
        row.get('filter_column'),
        row.get('filter_values')
    )

        # Charger la couche attribut depuis gpd_attributs
        print(f"Chargement de la couche: {attribute_name}")
        attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
        attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

        # Appliquer la méthode
        
        if method == "A": # Buffer feature extraction
            print(f"Applying method {method} for attribute {attribute_name}...")
            attribute_df = extract_buffer_feature(
                segments_gdf = segmented_net,
                feature_gdf = attribute_gdf,
                feature_name=attribute_name,
                geom_kind=geometry_type,
                buffer_radius=buffer_size,
                how=how,
                value_column=value_column,
                crs_meter_epsg=operation_crs,
                feature_query  = feature_query_expr,
            )
            print(f"Buffer feature extracted for {attribute_name} with method {method}")
        
                # Debug duplicates
            if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
                print("\nDEBUG: Found duplicate segment assignments")
                dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
                print(f"Number of segments with multiple zones: {len(dupes['segment_id'].unique())}")
            
                # Keep only the first occurrence for each segment_id
                attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
                print("Dropped duplicates, keeping first occurrence")
            print(f"Spatial join computed for {attribute_name} with method {method}") 
        
        # Ajoute d'autres méthodes si besoin
        
        # Ajouter la colonne au GeoDataFrame principal
        segmented_net[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name].fillna(0)
        print(f"Attribute {attribute_name} added to segmented_net.")

# Sauvegarder
print("Saving step 2 features to parquet...")
segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)


Boucle sur chaque attribut... peut prendre du temps (30 mn)
Chargement de la couche: accident
Applying method A for attribute accident...
Buffer feature extracted for accident with method A
Spatial join computed for accident with method A
Attribute accident added to segmented_net.
Chargement de la couche: zone_apaisee
Applying method A for attribute zone_apaisee...
Buffer feature extracted for zone_apaisee with method A
Spatial join computed for zone_apaisee with method A
Attribute zone_apaisee added to segmented_net.
Chargement de la couche: zone_pietonne
Applying method A for attribute zone_pietonne...
Buffer feature extracted for zone_pietonne with method A
Spatial join computed for zone_pietonne with method A
Attribute zone_pietonne added to segmented_net.
Chargement de la couche: vitesse
Applying method A for attribute vitesse...
Buffer feature extracted for vitesse with method A
Spatial join computed for vitesse with method A
Attribute vitesse added to segmented_net.
Chargement d

In [4]:

segmented_net.head(20)


,geometry,segment_id,length,accident_A_10,zone_apaisee_A_10,zone_pietonne_A_10,vitesse_A_10,stationnement_genant_A_10,connectivite_A_10,largeur_trottoir_A_10,topographie_A_10,eau_A_10,rez_actif_A_10,tp_A_200,amenite_A_10,espaces_ouverts_A_10,temperature_A_10,canopee_A_30,bruit_A_30
0,"LINESTRING (2505952.43 1117556.983, 2505957.52...",000000,18.611510,0.0,0.000000,0.000000,1.000000,0.0,16,2.0,3.1,0.0,0.0,3.0,0.0,0.0,32.117298,3.435069,0.949764
1,"LINESTRING (2504875.759 1116854.188, 2504891.4...",000001,17.373877,1.0,0.000000,0.000000,0.942774,0.0,252,4.0,5.6,0.0,0.0,5.0,0.0,1.0,33.267200,0.051274,4.000000
2,"LINESTRING (2500026.558 1117819.302, 2500041.9...",000002,50.000000,8.0,0.000000,0.060493,0.939451,6.0,963,9.0,62.9,0.0,2.0,9.0,2.0,4.0,33.157600,0.000000,3.652566
3,"LINESTRING (2500045.896 1117773.338, 2500047.9...",000003,7.198671,5.0,0.000000,0.055699,0.944288,1.0,637,7.0,21.9,0.0,0.0,8.0,0.0,2.0,0.000000,0.000000,3.399097
4,"LINESTRING (2498574.627 1115881.289, 2498530.0...",000004,45.125485,0.0,0.000000,0.501680,0.498278,0.0,621,10.0,49.6,0.0,1.0,4.0,1.0,2.0,32.843899,0.274643,2.243178
5,"LINESTRING (2503779.53 1116266.438, 2503790.87...",000005,11.749078,1.0,0.000000,0.143946,1.286035,0.0,98,3.0,12.2,0.0,0.0,7.0,0.0,0.0,31.430401,1.845129,1.000000
6,"LINESTRING (2496932.543 1119478.184, 2496913.2...",000006,22.405256,0.0,0.652118,0.000000,0.347882,0.0,24,6.0,27.3,0.0,0.0,2.0,0.0,1.0,0.000000,0.042198,2.069667
7,"LINESTRING (2504805.799 1117155.916, 2504813.8...",000007,12.111091,1.0,1.000000,0.000000,0.000000,0.0,7,0.0,10.2,0.0,0.0,0.0,0.0,0.0,29.096701,4.125688,1.000000
8,"LINESTRING (2498974.661 1122552.258, 2498967.7...",000008,18.010532,1.0,0.000000,0.000000,1.523508,0.0,48,0.0,49.3,0.0,0.0,4.0,0.0,0.0,0.000000,1.958039,1.000000
9,"LINESTRING (2499069.719 1114426.012, 2499064.2...",000009,14.050916,2.0,0.000000,0.000000,1.000000,0.0,156,7.0,36.7,0.0,0.0,8.0,0.0,2.0,32.784302,0.802692,2.199339


In [5]:
#import step2_features and convert to gpkg for QGIS use
segmented_net = gpd.read_parquet(os.path.join(output_step2_path, "step2_features.parquet"))
segmented_net.to_file(os.path.join(output_step2_path, "step2_features.gpkg"), driver="GPKG")
